<a href="https://colab.research.google.com/github/Calebchike/Bitcoin-EDA/blob/main/Bitcoin_EDA_2014_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Module**

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# **Import Data**

In [34]:
file_path = "/content/bitcoin_dataset.csv"
df = pd.read_csv(file_path)
df.head(5)

,Date,Open,High,Low,Close,Adj Close,Volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,398.821014,26580100


## **Duplicate dataframe and Set Date Column as index**

Create dataframe `bitcoin` a duplicate of the `df` dataframe.
Set the date column as the dataframe index.

In [35]:
bitcoin = df.copy()
bitcoin['Date'] = pd.to_datetime(df['Date'])
bitcoin.set_index('Date', inplace=True)
bitcoin.head(5)

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2014-09-17,465.864014,468.174011,452.421997,457.334015,457.334015,21056800
2014-09-18,456.859985,456.859985,413.104004,424.440002,424.440002,34483200
2014-09-19,424.102997,427.834991,384.532013,394.795990,394.795990,37919700
2014-09-20,394.673004,423.295990,389.882996,408.903992,408.903992,36863600
2014-09-21,408.084991,412.425995,393.181000,398.821014,398.821014,26580100


# **Question About the Data**

# **Price Behaviour**

## **Q1**. What was the total return from `2014` to `2026`, and what does $1,000 invested at the start become?

In [36]:
# Extract the first available starting price in 2014
price_2014 = bitcoin.loc['2014'].iloc[0]['Close']

# Extract the last available price in 2026
price_2026 = bitcoin.loc['2026'].iloc[-1]['Close']

# Calculate Total Return
total_return = ((price_2026 - price_2014) / price_2014) * 100

# Calculate Investment Growth
initial_investment = 1000
ending_value = initial_investment * (price_2026 / price_2014)

# Print results
print(f"Bitcoin Price (Start 2014): ${price_2014:,.2f}")
print(f"Bitcoin Price (Latest 2026): ${price_2026:,.2f}\n")
print(f"Total Return (2014-2026): {total_return:,.2f}%")
print(f"A $1,000 investment would become: ${ending_value:,.2f}")

Bitcoin Price (Start 2014): $457.33
Bitcoin Price (Latest 2026): $60,922.67

Total Return (2014-2026): 13,221.26%
A $1,000 investment would become: $133,212.63


## **Q2**. Which year had the best return? Which had the worst? By how much?

In [37]:
# Group by the year of the date index and get the first and last close price of each year
yearly_prices = bitcoin.groupby(bitcoin.index.year)['Close'].agg(['first', 'last'])

# Calculate the annual percentage return for each year
# Formula: ((End Price - Start Price) / Start Price) * 100
yearly_prices['Return (%)'] = ((yearly_prices['last'] - yearly_prices['first']) / yearly_prices['first']) * 100

# Identify the year with the best (maximum) return
best_year = yearly_prices['Return (%)'].idxmax()
best_return = yearly_prices['Return (%)'].max()

# Identify the year with the worst (minimum) return
worst_year = yearly_prices['Return (%)'].idxmin()
worst_return = yearly_prices['Return (%)'].min()

# Calculate the absolute difference between the best and worst annual returns
return_difference = best_return - worst_return

# Print the findings clearly
print(f"Best Year: {best_year} with a return of {best_return:,.2f}%")
print(f"Worst Year: {worst_year} with a return of {worst_return:,.2f}%")
print(f"The difference between the best and worst year is {return_difference:,.2f}%.")


Best Year: 2017 with a return of 1,318.02%
Worst Year: 2018 with a return of -72.60%
The difference between the best and worst year is 1,390.61%.


## **Q3**. How many days did it take to 10x, 100x, 200x from the starting price?

In [38]:
# Get the baseline starting price and its corresponding date
start_price = bitcoin['Close'].iloc[0]
start_date = bitcoin.index[0]

# Define the target milestones (10x, 100x, 200x)
multipliers = [10, 100, 200]

print(f"Baseline Start Date: {start_date.strftime('%Y-%m-%d')}")
print(f"Baseline Start Price: ${start_price:,.2f}\n")

# Iterate through each multiplier to find the time elapsed
for target in multipliers:
    # Calculate the dollar value needed to reach the milestone
    target_price = start_price * target

    # Filter the dataframe to only find days where the price met or exceeded the target
    milestone_df = bitcoin[bitcoin['Close'] >= target_price]

    # Check if Bitcoin has actually reached this milestone yet
    if not milestone_df.empty:
        # Extract the very first date this milestone condition was met
        achieved_date = milestone_df.index[0]

        # Calculate the delta between the milestone date and the start date
        days_taken = (achieved_date - start_date).days

        # Print the result
        print(f" {target}x Milestone (${target_price:,.2f}):")
        print(f"   - Reached on: {achieved_date.strftime('%Y-%m-%d')}")
        print(f"   - Time elapsed: {days_taken:,} days (approx. {days_taken/365.25:.1f} years)\n")
    else:
        print(f" {target}x Milestone (${target_price:,.2f}): Not reached yet in this dataset.\n")

Baseline Start Date: 2014-09-17
Baseline Start Price: $457.33

 10x Milestone ($4,573.34):
   - Reached on: 2017-08-29
   - Time elapsed: 1,077 days (approx. 2.9 years)

 100x Milestone ($45,733.40):
   - Reached on: 2021-02-08
   - Time elapsed: 2,336 days (approx. 6.4 years)

 200x Milestone ($91,466.80):
   - Reached on: 2024-11-19
   - Time elapsed: 3,716 days (approx. 10.2 years)



## **Q4**.  What is the largest single-day gain and the largest single-day loss, and what happened that day?

In [39]:
# Calculate the daily percentage change based on the closing price
bitcoin['Daily_Return'] = bitcoin['Close'].pct_change() * 100

# Extract the date and value for the absolute largest daily gain
max_gain_date = bitcoin['Daily_Return'].idxmax()
max_gain_value = bitcoin['Daily_Return'].max()

# Extract the date and value for the absolute largest daily loss
max_loss_date = bitcoin['Daily_Return'].idxmin()
max_loss_value = bitcoin['Daily_Return'].min()

# print the result
print(f" Largest Single-Day Gain: {max_gain_value:.2f}% on {max_gain_date.strftime('%Y-%m-%d')}")
print(f" Largest Single-Day Loss: {max_loss_value:.2f}% on {max_loss_date.strftime('%Y-%m-%d')}")

 Largest Single-Day Gain: 25.25% on 2017-12-07
 Largest Single-Day Loss: -37.17% on 2020-03-12


## **Q5**.  What percent of days closed green versus red? Is it close to a coin flip?

In [40]:
# Calculate daily returns (percentage change from previous day's close)
bitcoin['Daily_Return'] = bitcoin['Close'].pct_change() * 100

# Count the days based on performance conditions
green_days = (bitcoin['Daily_Return'] > 0).sum()
red_days = (bitcoin['Daily_Return'] < 0).sum()
flat_days = (bitcoin['Daily_Return'] == 0).sum()

# Calculate the total valid trading days (excluding the first row, which is NaN)
total_days = bitcoin['Daily_Return'].dropna().count()

# Compute the percentages
pct_green = (green_days / total_days) * 100
pct_red = (red_days / total_days) * 100
pct_flat = (flat_days / total_days) * 100

# Print the result
print(f" Green Days (Closed Up):   {green_days:,} ({pct_green:.2f}%)")
print(f" Red Days (Closed Down):   {red_days:,} ({pct_red:.2f}%)")
print(f" Flat Days (No Change):    {flat_days:,} ({pct_flat:.2f}%)")
print("-" * 45)
print(f"Total Evaluated Days:       {total_days:,}")

 Green Days (Closed Up):   2,242 (52.40%)
 Red Days (Closed Down):   2,036 (47.58%)
 Flat Days (No Change):    1 (0.02%)
---------------------------------------------
Total Evaluated Days:       4,279


# **Volatility and Risk**

##  **Q1**.   What is the rolling 30-day volatility, and which periods spike hardest?

In [41]:
# Calculate daily percentage returns
bitcoin['Daily_Return'] = bitcoin['Close'].pct_change() * 100

# Calculate the Standard Deviation of daily returns
bitcoin['Rolling_Vol_30'] = bitcoin['Daily_Return'].rolling(window=30).std()

bitcoin['Annualized_Vol_30'] = bitcoin['Rolling_Vol_30'] * np.sqrt(365)

# Identify the top 5 periods where volatility spiked the hardest
# We use dropna() to ignore the first 29 days which don't have enough data
top_vol_days = bitcoin['Rolling_Vol_30'].dropna().nlargest(5)

# Print the highest volatility spikes found in the dataset
print("Top 5 Dates with the Highest 30-Day Rolling Volatility Spikes:")
print("-" * 65)
for date, vol in top_vol_days.items():
    annualized = bitcoin.loc[date, 'Annualized_Vol_30']
    print(f"📅 {date.strftime('%Y-%m-%d')}: Daily Vol = {vol:.2f}% | Annualized Vol = {annualized:.2f}%")

Top 5 Dates with the Highest 30-Day Rolling Volatility Spikes:
-----------------------------------------------------------------
📅 2020-04-06: Daily Vol = 9.13% | Annualized Vol = 174.49%
📅 2020-04-03: Daily Vol = 9.06% | Annualized Vol = 173.05%
📅 2020-04-02: Daily Vol = 9.06% | Annualized Vol = 173.04%
📅 2020-04-10: Daily Vol = 9.05% | Annualized Vol = 172.95%
📅 2020-03-31: Daily Vol = 9.05% | Annualized Vol = 172.91%


##  **Q2**.   What is the largest drawdown from peak to trough, and how long did it take to recover?

In [42]:
# Compute a running, rolling maximum price up to each point in time
bitcoin['Rolling_Max'] = bitcoin['Close'].cummax()

# Calculate daily drawdown as the percentage distance from that rolling maximum
bitcoin['Drawdown'] = ((bitcoin['Close'] - bitcoin['Rolling_Max']) / bitcoin['Rolling_Max']) * 100

# Locate the worst drawdown value and its date
max_drawdown = bitcoin['Drawdown'].min()
trough_date = bitcoin['Drawdown'].idxmin()

# Find the exact date when the rolling max equaled the peak value at the trough
peak_value = bitcoin.loc[trough_date, 'Rolling_Max']
peak_date = bitcoin.loc[:trough_date][bitcoin.loc[:trough_date, 'Close'] == peak_value].index[0]

# Trace forward to find when the price fully recovered to breach that previous peak
recovery_df = bitcoin.loc[trough_date:][bitcoin.loc[trough_date:, 'Close'] >= peak_value]

if not recovery_df.empty:
    recovery_date = recovery_df.index[0]
    days_to_trough = (trough_date - peak_date).days
    days_to_recover = (recovery_date - trough_date).days
    total_cycle_days = (recovery_date - peak_date).days

    print(f"Maximum Drawdown: {max_drawdown:.2f}%")
    print(f"Peak Date:         {peak_date.strftime('%Y-%m-%d')} (${peak_value:,.2f})")
    print(f"Trough Date:       {trough_date.strftime('%Y-%m-%d')} (${bitcoin.loc[trough_date, 'Close']:,.2f})")
    print(f"Recovery Date:     {recovery_date.strftime('%Y-%m-%d')}")
    print("-" * 50)
    print(f"Days from Peak to Bottom (Crash Phase): {days_to_trough} days")
    print(f"Days from Bottom to Recovery (Rally Phase): {days_to_recover} days")
    print(f"Total Peak-to-Peak Recovery Cycle Timeline: {total_cycle_days} days")
else:
    print(f"Maximum Drawdown: {max_drawdown:.2f}%")
    print(f"Peak Date:         {peak_date.strftime('%Y-%m-%d')} (${peak_value:,.2f})")
    print(f"Trough Date:       {trough_date.strftime('%Y-%m-%d')}")
    print(f"Status:            Still in progress. The price has not yet recovered to its peak.")

Maximum Drawdown: -83.40%
Peak Date:         2017-12-16 ($19,497.40)
Trough Date:       2018-12-15 ($3,236.76)
Recovery Date:     2020-11-30
--------------------------------------------------
Days from Peak to Bottom (Crash Phase): 364 days
Days from Bottom to Recovery (Rally Phase): 716 days
Total Peak-to-Peak Recovery Cycle Timeline: 1080 days


##  **Q3**.    Does volume predict direction, or is the correlation near zero like the original found?

In [43]:
# Calculate Daily Price Return and Absolute (Magnitude) Return
bitcoin['Daily_Return'] = bitcoin['Close'].pct_change() * 100
bitcoin['Absolute_Return'] = bitcoin['Daily_Return'].abs()

# Calculate Daily Volume Percentage Change
# Using pct_change because raw volume grows exponentially over the years
bitcoin['Volume_Change'] = bitcoin['Volume'].pct_change() * 100

# Create a correlation matrix comparing Volume to both Directional and Absolute returns
correlation_matrix = bitcoin[['Volume_Change', 'Daily_Return', 'Absolute_Return']].corr()

print("--- Pearson Correlation Matrix ---")
print(correlation_matrix)


--- Pearson Correlation Matrix ---
                 Volume_Change  Daily_Return  Absolute_Return
Volume_Change         1.000000      0.033647         0.351137
Daily_Return          0.033647      1.000000         0.055939
Absolute_Return       0.351137      0.055939         1.000000


##  **Q4**.     Is volatility clustered — do high-volatility days follow other high-volatility days?

In [44]:
# Calculate daily returns and their absolute values (magnitude of volatility)
bitcoin['Daily_Return'] = bitcoin['Close'].pct_change() * 100
bitcoin['Absolute_Return'] = bitcoin['Daily_Return'].abs()

# Print day-to-day correlation
day_to_day_corr = bitcoin['Absolute_Return'].corr(bitcoin['Absolute_Return'].shift(1))
print(f"The day to day Correlation: {day_to_day_corr:.3f}")

The day to day Correlation: 0.211
